In [11]:
"""Elo Rating Calculator

Based on: https://www.omnicalculator.com/sports/elo
"""
import operator
from collections import defaultdict
import pandas as pd

def calculate_elo_rating(subject_elo_rating, agent_elo_rating, k_factor=20, score=1, number_of_decimals=1):
    """
    Calculates the Elo rating of a given subject given it's original score, it's opponent, 
    the K-Factor, and whether or not it has won or not. 
    The calculation is based on: https://www.omnicalculator.com/sports/elo

    Args:
        subject_elo_rating(float): The original Elo rating for the subject
        agent_elo_rating(float): The original Elo rating for the agent
        k_factor(int): k-factor, or development coefficient. 
            - It usually takes values between 10 and 40, depending on player's strength 
        score(int): the actual outcome of the game. 
            - In chess, a win counts as 1 point, a draw is equal to 0.5, and a lose gives 0.
        number_of_decimals(int): Number of decimals to round to
        
    Returns:
        int: Updated Elo rating of the subject
    """
    # Calculating the Elo rating
    rating_difference = agent_elo_rating - subject_elo_rating
    expected_score = 1 / (1 + 10 ** (rating_difference / 400))
    new_elo_rating = subject_elo_rating + k_factor * (score - expected_score)
    # Rounding to `number_of_decimals`
    return round(new_elo_rating, number_of_decimals)

def update_elo_rating(winner_id, loser_id, id_to_elo_rating=None, default_elo_rating=1000, \
    winner_score=1, loser_score=0, **calculate_elo_rating_params):
    """
    Updates the Elo rating in a dictionary that contains the ID of the subject as keys, 
    and the Elo rating as the values. You can also adjust how the Elo rating is calculated with 'calculate_elo_rating_params'.
    
    Args:
        winner_id(str): ID of the winner
        loser_id(str): ID of the loser
        id_to_elo_rating(dict): Dict that has the ID of the subjects as keys to the Elo Score as values
        default_elo_rating(int): The default Elo rating to be used if there is not elo score for the specified ID
        **calculate_elo_rating_params(kwargs): Other params for the calculate_elo_rating to change how the Elo rating is calculated
        
    Returns:
        Dict: Dict that has the ID of the subjects as keys to the Elo Score as values
    """
    if id_to_elo_rating is None:
        id_to_elo_rating = defaultdict(lambda:default_elo_rating)
    
    # Getting the current Elo Score
    current_winner_rating = id_to_elo_rating[winner_id] 
    current_loser_rating = id_to_elo_rating[loser_id] 
    
    # Calculating Elo rating            
    id_to_elo_rating[winner_id] = calculate_elo_rating(subject_elo_rating=current_winner_rating, \
        agent_elo_rating=current_loser_rating, score=winner_score, **calculate_elo_rating_params)
    id_to_elo_rating[loser_id] = calculate_elo_rating(subject_elo_rating=current_loser_rating, \
        agent_elo_rating=current_winner_rating, score=loser_score, **calculate_elo_rating_params)

    return id_to_elo_rating

def get_ranking_from_elo_rating_dictionary(input_dict, subject_id):
    """
    Orders a dictionary of subject ID keys to ELO score values by ELO score. 
    And then gets the rank of the subject with the inputted ID.
    Lower ranks like 1 would represent those subjects with higher ELO scores and vice versa.

    Args:
        input_dict(dict): 
            Dictionary of subject ID keys to ELO score values
        subject_id(str, int, or any value that's a key in input dict): 
            The ID of the subject that you want the ranking of

    Returns:
        int:
            Ranking of the subject with the ID inputted
    """
    # Sorting the subject ID's by ELO score
    sorted_subject_to_elo_rating = sorted(input_dict.items(), key=operator.itemgetter(1), reverse=True)
    # Getting the rank of the subject based on ELO score
    return [subject_tuple[0] for subject_tuple in sorted_subject_to_elo_rating].index(subject_id) + 1


def iterate_elo_rating_calculation_for_dataframe(dataframe, winner_id_column, loser_id_column, tie_column=None, additional_columns=None):
    """
    Iterates through a dataframe that has the ID of winners and losers for a given event. 
    A dictionary will be created that contains the information of the event, 
    which can then be turned into a dataframe. Each key is either from winner or loser's perspective. 

    Args:
        dataframe(Pandas DataFrame): 
        winner_id_column(str): The name of the column that has the winner's ID
        loser_id_column(str): The name of the column that has the loser's ID
        additional_columns(list): Additional columns to take from the 

    Returns:
        Dict: With a key value pair for each event either from the winner or loser's perspective. 
            This can be turned into a dataframe with each key value pair being a row.
    """
    if additional_columns is None:
        additional_columns = []

    # Dictionary that keeps track of the current Elo rating of the subject
    id_to_elo_rating = defaultdict(lambda:1000)
    # Dictionary that will be converted to a DataFrame
    index_to_elo_rating_and_meta_data = defaultdict(dict)

    # Indexes that will identify which row the dictionary key value pair will be
    # The number of the index has no significance other than being the number of the row
    all_indexes = iter(range(0, 99999))

    # Keeping track of the number of matches
    total_match_number = 1

    # Making a copy in case there is an error with changing the type of the tie column
    copied_dataframe = dataframe.copy()
    # Changing the tie column type to bool
    # So that we can filter out for booleans including False and 0
    try:
        copied_dataframe[tie_column] = copied_dataframe[tie_column].astype(bool)
    except:
        copied_dataframe = dataframe.copy()

    for index, row in copied_dataframe.dropna(subset=winner_id_column).iterrows():
        # Getting the ID of the winner subject
        winner_id = row[winner_id_column]
        # Getting the ID of the loser subject
        loser_id = row[loser_id_column]

        # Getting the current Elo Score
        current_winner_rating = id_to_elo_rating[winner_id] 
        current_loser_rating = id_to_elo_rating[loser_id] 

        if tie_column:
            # When there is nothing in the tie column
            # Or when there is a false value indicating that it is not a tie
            if pd.isna(copied_dataframe[tie_column][index]) or ~(copied_dataframe[tie_column][index]).any():
                winner_score = 1
                loser_score = 0
            # When there is value in the tie column
            else:
                winner_score = 0.5
                loser_score = 0.5
        # When there is no tie column
        else:
            winner_score = 1
            loser_score = 0

        # Updating the dictionary with ID keys and Elo Score values
        update_elo_rating(winner_id=winner_id, loser_id=loser_id, id_to_elo_rating=id_to_elo_rating, \
            winner_score=winner_score, loser_score=loser_score)

        # Saving all the data for the winner
        winner_index = next(all_indexes)
        index_to_elo_rating_and_meta_data[winner_index]["total_match_number"] = total_match_number
        index_to_elo_rating_and_meta_data[winner_index]["subject_id"] = winner_id
        index_to_elo_rating_and_meta_data[winner_index]["agent_id"] = loser_id
        index_to_elo_rating_and_meta_data[winner_index]["original_elo_rating"] = current_winner_rating
        index_to_elo_rating_and_meta_data[winner_index]["updated_elo_rating"] = id_to_elo_rating[winner_id]
        index_to_elo_rating_and_meta_data[winner_index]["win_draw_loss"] = winner_score
        index_to_elo_rating_and_meta_data[winner_index]["subject_ranking"] = get_ranking_from_elo_rating_dictionary(id_to_elo_rating, winner_id)
        index_to_elo_rating_and_meta_data[winner_index]["agent_ranking"] = get_ranking_from_elo_rating_dictionary(id_to_elo_rating, loser_id)
        index_to_elo_rating_and_meta_data[winner_index]["pairing_index"] = 0
        for column in additional_columns:
            index_to_elo_rating_and_meta_data[winner_index][column] = row[column]  

        # Saving all the data for the loser
        loser_index = next(all_indexes)
        index_to_elo_rating_and_meta_data[loser_index]["total_match_number"] = total_match_number
        index_to_elo_rating_and_meta_data[loser_index]["subject_id"] = loser_id
        index_to_elo_rating_and_meta_data[loser_index]["agent_id"] = winner_id
        index_to_elo_rating_and_meta_data[loser_index]["original_elo_rating"] = current_loser_rating
        index_to_elo_rating_and_meta_data[loser_index]["updated_elo_rating"] = id_to_elo_rating[loser_id]
        index_to_elo_rating_and_meta_data[loser_index]["win_draw_loss"] = loser_score
        index_to_elo_rating_and_meta_data[loser_index]["subject_ranking"] = get_ranking_from_elo_rating_dictionary(id_to_elo_rating, loser_id)
        index_to_elo_rating_and_meta_data[loser_index]["agent_ranking"] = get_ranking_from_elo_rating_dictionary(id_to_elo_rating, winner_id)
        index_to_elo_rating_and_meta_data[loser_index]["pairing_index"] = 1        
        for column in additional_columns:
            index_to_elo_rating_and_meta_data[loser_index][column] = row[column]  

        # Updating the match number
        total_match_number += 1

    return index_to_elo_rating_and_meta_data

In [12]:
import pandas as pd

# 1. Load your Excel file
file_path = r"C:\Users\sjs93\Downloads\c57_MF_3CA_Simmone.xlsx"
df = pd.read_excel(file_path)

# Check the first rows and column names
print(df.head())
print(df.columns)


    runner                 date      match  winner  loser  notes
0  Simmone  2025-09-14 00:00:00  1.1 V 1.2     1.2    1.1    NaN
1  Simmone                  NaN  1.3 V 1.4     1.4    1.3    NaN
2  Simmone                  NaN  1.1 V 1.3     1.1    1.3    NaN
3  Simmone                  NaN  1.2 V 1.4     1.4    1.2    NaN
4  Simmone                  NaN  1.4 V 1.1     1.4    1.1    NaN
Index(['runner', 'date', 'match', 'winner', 'loser', 'notes'], dtype='object')


In [13]:
import pandas as pd

# 1. Load your Excel file
file_path = r"C:\Users\sjs93\Downloads\c57_MF_3CA_Simmone.xlsx"
df = pd.read_excel(file_path)

# 2. Run Elo iteration using your winner/loser columns
results_dict = iterate_elo_rating_calculation_for_dataframe(
    dataframe=df,
    winner_id_column="winner",   # column in your file
    loser_id_column="loser",     # column in your file
    additional_columns=["runner", "date", "match", "notes"]  # keep metadata
)

# 3. Convert results to a DataFrame
elo_results_df = pd.DataFrame.from_dict(results_dict, orient="index")
print(elo_results_df.head())

# 4. Final Elo per subject
final_ratings = elo_results_df.groupby("subject_id")["updated_elo_rating"].last()
final_rankings = final_ratings.sort_values(ascending=False)
print("\nFinal Rankings:\n", final_rankings)

# 5. (Optional) Save to Excel
elo_results_df.to_excel(r"C:\Users\sjs93\Downloads\c57_MF_3CA_Simmone_elo.xlsx", index=False)
final_rankings.to_excel(r"C:\Users\sjs93\Downloads\c57_MF_3CA_Simmone_final_rankings.xlsx")


   total_match_number  subject_id  agent_id  original_elo_rating  \
0                   1         1.2       1.1               1000.0   
1                   1         1.1       1.2               1000.0   
2                   2         1.4       1.3               1000.0   
3                   2         1.3       1.4               1000.0   
4                   3         1.1       1.3                990.0   

   updated_elo_rating  win_draw_loss  subject_ranking  agent_ranking  \
0              1010.0              1                1              2   
1               990.0              0                2              1   
2              1010.0              1                2              4   
3               990.0              0                4              2   
4              1000.0              1                3              4   

   pairing_index   runner                 date      match  notes  
0              0  Simmone  2025-09-14 00:00:00  1.1 V 1.2    NaN  
1              1  Simmon

In [14]:
import pandas as pd

file_path = r"C:\Users\sjs93\Downloads\c57_MF_3CA_Simmone.xlsx"

# Get all sheet names
sheet_names = pd.ExcelFile(file_path).sheet_names
print("Sheet names:", sheet_names)


Sheet names: ['Cage #1-Males', 'Cage #2-Females']


In [15]:
# --- Cage #1 (Males) ---
df_males = pd.read_excel(file_path, sheet_name="Cage #1-Males")

results_males = iterate_elo_rating_calculation_for_dataframe(
    dataframe=df_males,
    winner_id_column="winner",
    loser_id_column="loser",
    additional_columns=["runner", "date", "match", "notes"]
)
elo_males_df = pd.DataFrame.from_dict(results_males, orient="index")

final_males = elo_males_df.groupby("subject_id")["updated_elo_rating"].last().sort_values(ascending=False)
print("\nCage #1 (Males) Rankings:\n", final_males)


# --- Cage #2 (Females) ---
df_females = pd.read_excel(file_path, sheet_name="Cage #2-Females")

results_females = iterate_elo_rating_calculation_for_dataframe(
    dataframe=df_females,
    winner_id_column="winner",
    loser_id_column="loser",
    additional_columns=["runner", "date", "match", "notes"]
)
elo_females_df = pd.DataFrame.from_dict(results_females, orient="index")

final_females = elo_females_df.groupby("subject_id")["updated_elo_rating"].last().sort_values(ascending=False)
print("\nCage #2 (Females) Rankings:\n", final_females)



Cage #1 (Males) Rankings:
 subject_id
1.4    1118.1
1.1    1028.8
1.2     970.7
1.3     882.4
Name: updated_elo_rating, dtype: float64

Cage #2 (Females) Rankings:
 subject_id
2.3    1119.0
2.2    1038.4
2.4     961.4
2.1     881.2
Name: updated_elo_rating, dtype: float64


In [16]:
# Combine and save as CSV
final_results = pd.concat(
    [final_males.rename("Elo_Rating").to_frame().assign(Cage="Cage1_Males"),
     final_females.rename("Elo_Rating").to_frame().assign(Cage="Cage2_Females")]
)
final_results.to_csv("elo_rankings.csv", index=True)

print("✅ Elo rankings saved to 'elo_rankings.csv'")


✅ Elo rankings saved to 'elo_rankings.csv'


In [ ]:
final_results
